In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
train_df = pd.read_csv('/kaggle/input/datasets/amansingh45/diabetes/train.csv')
test_df = pd.read_csv('/kaggle/input/datasets/amansingh45/diabetes/test.csv')

In [ ]:
X = train_df.drop(["id", "label"], axis=1).values
y = train_df["label"].values

In [ ]:
X_test_final = test_df.drop("id", axis=1).values
test_ids = test_df["id"]

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test_final = scaler.transform(X_test_final)

In [ ]:
X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)
X_test_final = torch.FloatTensor(X_test_final)

y_train = torch.FloatTensor(y_train).view(-1,1)
y_valid = torch.FloatTensor(y_valid).view(-1,1)

input_size = X_train.shape[1]

In [ ]:
hidden_sizes = [16,32,64]
learning_rates = [0.01,0.001]

epochs = 100

results = []

In [ ]:
def accuracy(pred,label):
    pred = (pred>=0.5).float()
    return (pred==label).float().mean().item()

for hidden in hidden_sizes:

    for lr in learning_rates:

        model = nn.Sequential(
            nn.Linear(input_size, hidden),
            nn.ReLU(),

            nn.Linear(hidden, hidden//2),
            nn.ReLU(),

            nn.Linear(hidden//2,1),
            nn.Sigmoid()
        )

        criterion = nn.BCELoss()

        optimizer = optim.Adam(model.parameters(),lr=lr)

        for epoch in range(epochs):

            model.train()

            optimizer.zero_grad()

            outputs = model(X_train)

            loss = criterion(outputs,y_train)

            loss.backward()

            optimizer.step()

        model.eval()

        with torch.no_grad():

            train_pred = model(X_train)
            valid_pred = model(X_valid)

            train_acc = accuracy(train_pred,y_train)
            valid_acc = accuracy(valid_pred,y_valid)

        results.append({
            "Hidden":hidden,
            "LR":lr,
            "TrainAcc":train_acc,
            "TestAcc":valid_acc,
            "Model":model
        })

        print(f"Hidden={hidden} LR={lr} Train={train_acc:.4f} Test={valid_acc:.4f}")


In [ ]:
results_df = pd.DataFrame(results)

print(results_df[["Hidden","LR","TrainAcc","TestAcc"]])

In [ ]:
best = results_df.iloc[results_df["TestAcc"].idxmax()]

print("\nBest Hyperparameters")
print(best[["Hidden","LR","TrainAcc","TestAcc"]])

best_model = best["Model"]

In [ ]:
plt.figure(figsize=(8,5))

for lr in learning_rates:

    temp = results_df[results_df["LR"]==lr]

    plt.plot(temp["Hidden"],temp["TrainAcc"],marker='o',label=f"Train LR={lr}")

    plt.plot(temp["Hidden"],temp["TestAcc"],marker='s',label=f"Test LR={lr}")

plt.xlabel("Hidden Layer Size")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Hidden Layer Size")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

for hidden in hidden_sizes:

    temp = results_df[results_df["Hidden"]==hidden]

    plt.plot(temp["LR"],temp["TrainAcc"],marker='o',label=f"Train Hidden={hidden}")

    plt.plot(temp["LR"],temp["TestAcc"],marker='s',label=f"Test Hidden={hidden}")

plt.xlabel("Learning Rate")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Learning Rate")
plt.xscale("log")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
best_model.eval()

with torch.no_grad():
    probs = best_model(X_test_final)

predictions = (probs >= 0.5).int().numpy().flatten()

submission = pd.DataFrame({
    "id": test_ids,
    "label": predictions
})

submission.to_csv("submission.csv", index=False)

print("\nsubmission.csv generated successfully!")
print(submission.head())